In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score

# ---------------------------
# Step 1: Original dataset
data = {
    "Date": ["01-01-2026","02-01-2026","03-01-2026","04-01-2026","05-01-2026","06-01-2026"],
    "Temperature": [25, 30, 28, np.nan, 100, 22],
    "Humidity": [60, 65, np.nan, 70, 75, 80],
    "WindSpeed": [10, np.nan, 12, 14, 13, 200],
    "Precipitation": [2, 5, 0, 1, np.nan, 3],
    "Pressure": [1012, 1015, 1010, np.nan, 1020, 5000],
    "WeatherCondition": ["Sunny", "Cloudy", "Rainy", "Sunny", np.nan, "Rainy"]
}

df = pd.DataFrame(data)

# ---------------------------
# Step 2: Handle Missing Values
num_cols = ["Temperature", "Humidity", "WindSpeed", "Precipitation", "Pressure"]
df[num_cols] = df[num_cols].fillna(df[num_cols].mean())
df["WeatherCondition"] = df["WeatherCondition"].fillna(df["WeatherCondition"].mode()[0])

# ---------------------------
# Step 3: Handle Outliers using IQR
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = np.where(df[col] < lower, lower, df[col])
    df[col] = np.where(df[col] > upper, upper, df[col])

print("Cleaned Dataset:")
print(df)
print("\n")

# ---------------------------
# Step 4: Prepare data for regression
# For regression models: predict Temperature
X = df.drop(["Date", "Temperature", "WeatherCondition"], axis=1)
y = df["Temperature"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ---------------------------
# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Lasso Regression
lasso = Lasso(alpha=0.1)
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)

print("Linear Regression:")
print("MAE:", mean_absolute_error(y_test, y_pred_lr))
print("MSE:", mean_squared_error(y_test, y_pred_lr))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr)))
print("R2 Score:", r2_score(y_test, y_pred_lr))
print("\nLasso Regression:")
print("MAE:", mean_absolute_error(y_test, y_pred_lasso))
print("MSE:", mean_squared_error(y_test, y_pred_lasso))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lasso)))
print("R2 Score:", r2_score(y_test, y_pred_lasso))

# ---------------------------
# Logistic Regression (predict Rainy vs Not Rainy)
df_class = pd.get_dummies(df, columns=["WeatherCondition"], drop_first=True)
X_clf = df_class.drop(["Date", "Temperature", "WeatherCondition_Rainy"], axis=1)
y_clf = df_class["WeatherCondition_Rainy"]

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)

logr = LogisticRegression(max_iter=1000)
logr.fit(X_train_clf, y_train_clf)
y_pred_log = logr.predict(X_test_clf)

print("\nLogistic Regression Accuracy:", accuracy_score(y_test_clf, y_pred_log))

Cleaned Dataset:
         Date  Temperature  Humidity  WindSpeed  Precipitation  Pressure  \
0  01-01-2026         25.0      60.0      10.00            2.0   1012.00   
1  02-01-2026         30.0      65.0      49.80            5.0   1015.00   
2  03-01-2026         28.0      70.0      12.00            0.0   1010.00   
3  04-01-2026         41.0      70.0      14.00            1.0   1811.40   
4  05-01-2026         57.0      75.0      13.00            2.2   1020.00   
5  06-01-2026         22.0      80.0      83.75            3.0   2514.75   

  WeatherCondition  
0            Sunny  
1           Cloudy  
2            Rainy  
3            Sunny  
4            Rainy  
5            Rainy  


Linear Regression:
MAE: 50.76314235947595
MSE: 2693.9535496036287
RMSE: 51.90330962090596
R2 Score: -430.0325679365806

Lasso Regression:
MAE: 47.59270265673936
MSE: 2372.096134970405
RMSE: 48.704169585061244
R2 Score: -378.53538159526477


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.030e-01, tolerance: 7.220e-02
  model = cd_fast.enet_coordinate_descent(



Logistic Regression Accuracy: 0.0
